In [ ]:

import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor

from sklearn.model_selection import cross_validate, GroupShuffleSplit, GroupKFold

from sklearn.pipeline import make_pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.linear_model import LinearRegression, Ridge, Lasso
 



In [76]:
path = '../data/processed/clean.csv'

df = pd.read_csv(path)

In [77]:
X = df.drop(columns=['price','id'])
y = df['price']
groups = df['id']

In [78]:
print(X.shape)
print(y.shape)

(21613, 17)
(21613,)


In [79]:
gss = GroupShuffleSplit(test_size=0.2,random_state=42)

In [80]:
train_idx, test_idx = next(gss.split(X, y, groups=groups))

In [81]:
X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(17289, 17) (4324, 17)
(17289,) (4324,)


#baseline model

In [82]:

dummy_cross = cross_validate(
    DummyRegressor(), X_train, y_train,
    cv=GroupKFold(n_splits=5),
    groups=groups_train,
    scoring="neg_root_mean_squared_error"
)


print(f'baseline rsme:{-dummy_cross["test_score"].mean()}')

baseline rsme:365047.28943794535


#Linear regression model with all features

In [83]:

linear_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

linear_cross = cross_validate(
    linear_pipe, X_train, y_train,
    cv=GroupKFold(n_splits=5),
    groups=groups_train,
    scoring="neg_root_mean_squared_error",
    return_train_score=True
)

train_scores = -linear_cross['train_score']
test_scores = -linear_cross["test_score"]

print(f'train_score:{train_scores.mean()}')
print(f'test_score {test_scores.mean()}')


train_score:201231.68864150112
test_score 201744.7254977328


## Finding: Cv  on raw price  

| Model | Train RMSE | Validation RMSE | Typical miss vs median (450k) |
|---|---|---|---|
| Baseline | - | about 365k | about 81% |
| Linear regression | about 201k | about 201k | about 45% |

Train and validation are close. There is no overftting. 

#log the price

In [84]:
import numpy as np

In [85]:
print(f'price without log {y_train.skew()}')
print(f'price with log {np.log1p(y_train).skew()}')

price without log 3.997641358024732
price with log 0.4322786020392791


##baseline with logged price

In [86]:
y_train_log = np.log1p(y_train)

dummy_cross_log = cross_validate(
  DummyRegressor(),X_train,y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error"
  
)

print(f'baseline  with logged price:{-dummy_cross_log['test_score'].mean()}')

baseline  with logged price:0.5255715434140307


## linear regression with logged price

In [87]:
linear_pipe_log = make_pipeline(
 SimpleImputer(strategy="median"),
  LinearRegression()
)

linear_cross_log = cross_validate(
  linear_pipe_log,X_train, y_train_log,
  cv = GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
  

)


print(f'train_score_log:{-linear_cross_log['train_score'].mean()}')

print(f"test_score_log:{-linear_cross_log['test_score'].mean()}")

train_score_log:0.25334952157753454
test_score_log:0.25368227027130674


## Finding: CV on log price  

| Target | Model | Train RMSE | Validation RMSE |
|---|---|---|---|
| log price | Baseline | - | 0.526 |
| log price | Linear | 0.253 | 0.254 |

There is sign of  overfitting. Linear halves the baseline error. A typical miss of about 29% (factor 1.29) leaves room to improvement.


## Feature Engineering & Feature Selection


In [88]:
diff = X_train["sqft_living"] - X_train["sqft_above"] - X_train["sqft_basement"]

print(diff.abs().max())
print((diff != 0).sum())

0
0


In [ ]:
X_train_a = X_train.drop(columns=["sqft_living"])

linear_pipe_a = make_pipeline(
  SimpleImputer(strategy="median"),
  LinearRegression()
)

linear_cross_a = cross_validate(
  linear_pipe_a,X_train_a,y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print(f"train score : {-linear_cross_a['train_score'].mean()}")

print(f"test score: {-linear_cross_a['test_score'].mean()}")

train score : 0.2533495215775346
test score: 0.25368227027130913



sqft_living equals sqft_above + sqft_basement on every training row so removing  it was removed.

| Features | Train RMSE | Validation RMSE |
|---|---|---|
| All features | 0.2533 | 0.2537 |
| Without sqft_living | 0.2533 | 0.2537 |
 

In [90]:
X_train = X_train_a

In [91]:
X_train.shape

(17289, 16)

## feature ablation for all features

In [92]:
results = {}

for m in X_train.columns:
    X_drop = X_train.drop(columns=m)

    pipe = make_pipeline(
        SimpleImputer(strategy="median"),
        LinearRegression()
    )

    cv_out = cross_validate(
        pipe, X_drop, y_train_log,
        cv=GroupKFold(n_splits=5),
        groups=groups_train,
        scoring="neg_root_mean_squared_error"
    )

    results[m] = -cv_out["test_score"].mean()

print(pd.Series(results).sort_values())

long             0.253708
sqft_lot15       0.253727
bedrooms         0.253778
sqft_lot         0.253959
renovated        0.254003
floors           0.254877
bathrooms        0.255350
waterfront       0.255495
view             0.256367
condition        0.256900
sqft_living15    0.257016
sqft_basement    0.257668
sqft_above       0.258778
yr_built         0.260902
grade            0.272916
lat              0.310057
dtype: float64


In [93]:
drop_cols = ["long", "sqft_lot15", "bedrooms"]

X_dropped = X_train.drop(columns=drop_cols)



In [94]:
linear_pipe_droped = make_pipeline(
  SimpleImputer(strategy="median"),
  LinearRegression(),
)

linear_cross_dropped = cross_validate(
  linear_pipe_droped,X_dropped,y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True


)

print(f"train score : {-linear_cross_dropped['train_score'].mean()}")
print(f"test score: {-linear_cross_dropped['test_score'].mean()}")

train score : 0.2535877063604886
test score: 0.2538515385484388


Dropping the three weakest features together changes almost nothing. All 16 features are still kept

| Features | Train RMSE | Validation RMSE |
|---|---|---|
| All 16 | 0.2534 | 0.2537 |
| Without long, sqft_lot15, bedrooms | 0.2536 | 0.2539 |


### check for outliners

In [95]:
print(X_train.describe().T[["min", "max"]])

                     min           max
bedrooms          1.0000  1.100000e+01
bathrooms         0.5000  8.000000e+00
sqft_lot        520.0000  1.651359e+06
floors            1.0000  3.500000e+00
waterfront        0.0000  1.000000e+00
view              0.0000  4.000000e+00
condition         1.0000  5.000000e+00
grade             3.0000  1.300000e+01
sqft_above      384.0000  9.410000e+03
sqft_basement     0.0000  4.820000e+03
yr_built       1900.0000  2.015000e+03
lat              47.1559  4.777760e+01
long           -122.5190 -1.213150e+02
sqft_living15   399.0000  6.210000e+03
sqft_lot15      651.0000  8.712000e+05
renovated         0.0000  1.000000e+00


In [96]:
print(X_train["sqft_lot"].quantile([0.5, 0.9, 0.99, 1.0]))

0.50       7600.00
0.90      21359.60
0.99     210882.32
1.00    1651359.00
Name: sqft_lot, dtype: float64


In [97]:
big = X_train["sqft_lot"] > 210882
print(big.sum())
print(y_train[big].median(), y_train.median())

173
579000.0 450000.0


In [98]:
X_log = X_train.copy()
X_log["sqft_lot"] = np.log1p(X_log["sqft_lot"])
X_log["sqft_lot15"] = np.log1p(X_log["sqft_lot15"])

In [99]:
linear_pipe_xlog = make_pipeline(
  SimpleImputer(strategy="median"),
  LinearRegression(),
)

linear_cross_xlog = cross_validate(
  linear_pipe_xlog,X_log,y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True


)

print(f"train score : {-linear_cross_xlog['train_score'].mean()}")
print(f"test score: {-linear_cross_xlog['test_score'].mean()}")

train score : 0.2528897782447724
test score: 0.2532121514797875


In [100]:
old = -linear_cross_log["test_score"]
new = -linear_cross_xlog["test_score"]

print(old.round(4))
print(new.round(4))
print((old - new).round(4))

[0.2559 0.25   0.251  0.2595 0.252 ]
[0.2551 0.2494 0.251  0.2589 0.2517]
[ 0.0008  0.0006 -0.      0.0006  0.0003]




| Features | Train RMSE | Validation RMSE |
|---|---|---|
| 16 features, raw lot columns | 0.2534 | 0.2537 |
| 16 features, log of lot columns | 0.2529 | 0.2532 |

The lot columns have a long right tail,about 1% of rows, priced a bit higher, so we keep and the scale with log1p. 

In [101]:
X_train = X_log

##Ridge modelling

In [102]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [103]:
ridge_pipe_01 = make_pipeline(
  SimpleImputer(strategy="median"),
  StandardScaler(),
  Ridge(0.1)
)

ridge_cross = cross_validate(
  ridge_pipe_01,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -ridge_cross["train_score"].mean())

print("Validation RMSE:", -ridge_cross["test_score"].mean())

Train RMSE: 0.25288977825905024
Validation RMSE: 0.2532121430640144


In [104]:
ridge_pipe = make_pipeline(
  SimpleImputer(strategy="median"),
  StandardScaler(),
  Ridge()
)

ridge_cross = cross_validate(
  ridge_pipe,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -ridge_cross["train_score"].mean())

print("Validation RMSE:", -ridge_cross["test_score"].mean())

Train RMSE: 0.2528897796719437
Validation RMSE: 0.25321206864732243


In [105]:
ridge_pipe_10 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=10)
)


ridge_cross_10 = cross_validate(
  ridge_pipe_10,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -ridge_cross_10["train_score"].mean())

print("Validation RMSE:", -ridge_cross_10["test_score"].mean())

Train RMSE: 0.2528899203494653
Validation RMSE: 0.253211455044535


In [106]:
ridge_pipe_100 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=100)
)


ridge_cross_100 = cross_validate(
  ridge_pipe_100,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -ridge_cross_100["train_score"].mean())

print("Validation RMSE:", -ridge_cross_100["test_score"].mean())

Train RMSE: 0.2529034165616328
Validation RMSE: 0.25321778185196087


### Ridge Regression Findings

I tested alpha values of 0.1, 1, 10, and 100 using 5-fold GroupKFold cross-validation. Validation RMSE remained approximately 0.2537 across all values. Ridge Regression did not meaningfully improve on Linear Regression

## Lasso Regression

In [107]:
from sklearn.linear_model import Lasso

In [108]:
lasso_pipe_001 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso(alpha=0.01)
)



lasso_cross_001 = cross_validate(
  lasso_pipe_001,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -lasso_cross_001["train_score"].mean())

print("Validation RMSE:", -lasso_cross_001["test_score"].mean())

Train RMSE: 0.2550375817327896
Validation RMSE: 0.25527791730091526


In [109]:
lasso_pipe_0001 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso(alpha=0.001)
)



lasso_cross_0001 = cross_validate(
  lasso_pipe_0001,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -lasso_cross_0001["train_score"].mean())

print("Validation RMSE:", -lasso_cross_0001["test_score"].mean())

Train RMSE: 0.25296806602426386
Validation RMSE: 0.2532770632681297


In [110]:
lasso_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso()
)



lasso_cross = cross_validate(
  lasso_pipe,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -lasso_cross["train_score"].mean())

print("Validation RMSE:", -lasso_cross["test_score"].mean())

Train RMSE: 0.5255636250273265
Validation RMSE: 0.5255715434140307


### Lasso Regression Findings

Lasso performed poorly with alpha = 1 but improved as alpha decreased. Alpha = 0.001 gave the best validation RMSE of 0.2533.i can conclude that Lasso did not meaningfully improve on Linear Regression too

## polynomial 

In [113]:
from sklearn.preprocessing import PolynomialFeatures



In [115]:
poly_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PolynomialFeatures(degree=2),
    LinearRegression()
)



poly_cross = cross_validate(
  poly_pipe,
  X_train,
  y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
)

print("Train RMSE:", -poly_cross["train_score"].mean())

print("Validation RMSE:", -poly_cross["test_score"].mean())

Train RMSE: 0.22101908676343948
Validation RMSE: 0.2243041683371966


### Polynomial Regression Findings

Degree-2 Polynomial Regression improved validation RMSE to 0.2243, outperforming Linear Regression.

Degree 2 is our best model so far. Next, we’ll test it on the test set.

In [116]:
X_test_final = X_test.drop(columns=["sqft_living"]).copy()

X_test_final["sqft_lot"] = np.log1p(X_test_final["sqft_lot"])

X_test_final["sqft_lot15"] = np.log1p(X_test_final["sqft_lot15"])

In [120]:

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PolynomialFeatures(degree=2),
    LinearRegression()
)

model.fit(X_train, y_train_log)


pred_log = model.predict(X_test_final)
pred_price = np.expm1(pred_log)


from sklearn.metrics import mean_absolute_error, root_mean_squared_error



test_rmse_log = root_mean_squared_error(
    np.log1p(y_test),
    pred_log
)

print("Test RMSE (log price):", test_rmse_log)

print("Test MAE:", mean_absolute_error(y_test, pred_price))

print("Test RMSE:", root_mean_squared_error(y_test, pred_price))

Test RMSE (log price): 0.22720381121614042
Test MAE: 96466.69089810108
Test RMSE: 163840.37001235998


### Final Test Evaluation

The degree-2 Polynomial Regression model achieved a test RMSE of 0.2272 on the log-price scale, close to its cross-validation RMSE of 0.2243. The test MAE was approximately $96,467.

**Conclusion:** The model generalizes reasonably well to unseen data and is our current best candidate.

#testing

In [1]:
import sys
sys.path.append("../src")
from predict import predict_house_price

predict_house_price("../data/raw/housing.csv")[["price", "predicted_price"]].head()

,price,predicted_price
0,221900.0,296066.440689
1,538000.0,576093.515432
2,180000.0,428241.938316
3,604000.0,382903.305366
4,510000.0,471606.000028
